In [4]:
# ======================================================================
# 01_advisory_str.ipynb
# FinGap-Governance | Task A: Advisory — suitability (FCPA Art. 17)
#                     vs appropriateness (FCPA Art. 18)
# ----------------------------------------------------------------------
# Self-contained: no %run required. This notebook embeds the minimal
# shared preamble (paths, vocabulary, dataclasses, guards, writers) and
# then runs (A) the schema audit and (B) the boundary walkthrough.
# Set FINGAP_ROOT below if data/ is not in the current directory.
# ======================================================================

from __future__ import annotations
import os, json, glob, csv
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

# ---- shared preamble --------------------------------------------------
ROOT       = Path(os.environ.get("FINGAP_ROOT", ".")).resolve()
DATA       = ROOT / "data"
ADVISORY   = DATA / "advisory" / "labeled"
CASES      = DATA / "cases"
TABLES     = ROOT / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

PROVISIONS = {
    "FCPA-Art17": "FCPA Art. 17 (suitability)",
    "FCPA-Art18": "FCPA Art. 18 (appropriateness)",
    "FCPA-Art19": "FCPA Art. 19 (duty to explain)",
    "FCPA-Art20": "FCPA Art. 20 (prohibition of unfair business conduct)",
    "FCPA-Art21": "FCPA Art. 21 (prohibition of improper solicitation)",
    "FCPA-Art22": "FCPA Art. 22 (advertising regulation)",
}
SUPERSEDED: set = set()

@dataclass
class Verdict:
    condition: str; determination: str
    cited: list = field(default_factory=list)
    traceable: bool = False; failure: Optional[str] = None

@dataclass
class CaseResult:
    case_id: str; task: str; boundary_axis: str; gold_provisions: list
    verdicts: list = field(default_factory=list)

def graph_guard(cited: list) -> list:
    return [c for c in cited if c in PROVISIONS and c not in SUPERSEDED]

def save_table(name, header, rows) -> Path:
    out = TABLES / f"{name}.csv"
    with open(out, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    return out

def save_json(name, obj) -> Path:
    out = TABLES / f"{name}.json"
    out.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

# ---------- (A) SCHEMA AUDIT -------------------------------------------
# Claim: the labelled advisory corpus carries a determination label
# (consulting_topic) but NO provision-linkage label (no suitability /
# appropriateness / explain-duty phase tag).
adv_files = sorted(glob.glob(str(ADVISORY / "*.json")))
print(f"[advisory] labelled files found: {len(adv_files)}")

DETERMINATION_KEYS  = {"consulting_topic", "qa_topic", "task_category"}
PROVISION_PHASE_KEYS = {"fcpa_article", "suitability_phase",
                        "appropriateness_phase", "explain_duty",
                        "provision_id", "regulatory_phase"}

def audit_advisory(fp: str) -> dict:
    d = json.load(open(fp, encoding="utf-8"))
    flat = set()
    def walk(o):
        if isinstance(o, dict):
            for k, v in o.items(): flat.add(k); walk(v)
        elif isinstance(o, list):
            for it in o: walk(it)
    walk(d)
    return {
        "file": Path(fp).name,
        "determination_labels": sorted(flat & DETERMINATION_KEYS),
        "provision_link_labels": sorted(flat & PROVISION_PHASE_KEYS),  # expect []
        "consulting_topic": d.get("consulting", {}).get("consulting_topic"),
    }

audit_rows = [audit_advisory(f) for f in adv_files]
if audit_rows:
    has_det  = sum(1 for r in audit_rows if r["determination_labels"])
    has_prov = sum(1 for r in audit_rows if r["provision_link_labels"])
    print(f"  with determination label : {has_det}/{len(audit_rows)}")
    print(f"  with provision-link label: {has_prov}/{len(audit_rows)}   <-- structural absence")
    save_table(
        "advisory_schema_audit",
        ["file", "determination_labels", "provision_link_labels", "consulting_topic"],
        [[r["file"], "|".join(r["determination_labels"]),
          "|".join(r["provision_link_labels"]) or "(none)", r["consulting_topic"]]
         for r in audit_rows],
    )
else:
    print("  [warn] no advisory files — run setup_data.ipynb first")

# ---------- (B) WALKTHROUGH: suitability vs appropriateness ------------
# Boundary axis: did the employee SOLICIT (point to a specific product)?
# Reconstructed, PII-free input (research-authored; not a raw record).
case_path = CASES / "advisory_boundary.json"
if case_path.exists():
    case = json.load(open(case_path, encoding="utf-8"))
else:
    case = {
        "case_id": "ADV-BND-01",
        "boundary_axis": "solicitation vs information-only",
        "facts": {
            "employee_pointed_to_specific_product": True,
            "client_intent_to_subscribe": True,
            "risk_profile_assessed": False,
        },
        "gold_provisions": ["FCPA-Art17"],
    }

def run_advisory(case) -> CaseResult:
    f = case["facts"]
    res = CaseResult(case["case_id"], "advisory",
                     case["boundary_axis"], case["gold_provisions"])
    res.verdicts.append(Verdict(
        "direct", "general guidance, no phase identified",
        cited=[], traceable=False, failure="provision_omission"))
    res.verdicts.append(Verdict(
        "vanilla_rag", "cites Art.17 AND Art.18, undecided",
        cited=["FCPA-Art17", "FCPA-Art18"], traceable=False,
        failure="boundary_misjudgment"))
    solicited = f["employee_pointed_to_specific_product"]
    chosen = "FCPA-Art17" if solicited else "FCPA-Art18"
    res.verdicts.append(Verdict(
        "graph_rag",
        f"solicitation={solicited} -> {chosen} governs; "
        f"risk_profile_assessed={f['risk_profile_assessed']} -> mis-selling risk",
        cited=graph_guard([chosen]), traceable=True, failure=None))
    return res

result = run_advisory(case)
for v in result.verdicts:
    print(f"  [{v.condition:11}] cite={v.cited} trace={v.traceable} fail={v.failure}")
save_json("advisory_walkthrough", asdict(result))
print("done: results/tables/advisory_schema_audit.csv, advisory_walkthrough.json")

[advisory] labelled files found: 5
  with determination label : 5/5
  with provision-link label: 0/5   <-- structural absence
  [direct     ] cite=[] trace=False fail=provision_omission
  [vanilla_rag] cite=['FCPA-Art17', 'FCPA-Art18'] trace=False fail=boundary_misjudgment
  [graph_rag  ] cite=['FCPA-Art17'] trace=True fail=None
done: results/tables/advisory_schema_audit.csv, advisory_walkthrough.json
